In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_api_key = os.environ["HUGGINGFACE_API_KEY"]

In [5]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cpu" # the device to load the model onto

In [6]:

model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", token=hf_api_key)
model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )
    (norm): MistralRMSNorm(

In [7]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", token=hf_api_key)

In [8]:
import pymongo
from pymongo import MongoClient
import matplotlib.pyplot as plt
import numpy as np

client = MongoClient('localhost', 5001)

db = client.dataset_db
annotation = db.fine_tuning

In [9]:
import pandas as pd

query = {}

cursor = annotation.find(query)

activity_df = pd.DataFrame(list(cursor))

/tmp/ipykernel_2048473/144681116.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [17]:
activity_df

before_text = activity_df.iloc[0]["before_text"]
after_text = activity_df.iloc[0]["after_text"]
print(before_text)

\section{Constructing ``TenPageStories'' Dataset}\label{sec:app:ten_page_stories}
\preston{talk about the construction of the dataset in detail. Iterative process. Prompts used for three settings, etc.}


In [23]:
generate_text = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    return_full_text=False,  # if using langchain set True
    task="text-generation",
    # we pass model parameters here too
    temperature=0.1,  # 'randomness' of outputs, 0.0 is the min and 1.0 the max
    top_p=0.15,  # select from top tokens whose probability add up to 15%
    top_k=0,  # select from top 0 tokens (because zero, relies on top_p)
    max_new_tokens=4000,  # max number of tokens to generate in the output
    repetition_penalty=1.1  # if output begins repeating increase
)

In [24]:
res = generate_text(before_text)
print(res[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
print(res)

[{'generated_text': '\n\n\\subsection{Prompts and Settings}'}]
